# STEP 2: LangChain 기초

이 노트북에서는 로컬 Ollama의 `qwen3.6:27b`를 LangChain으로 호출합니다.

학습 순서: `Chat Model 호출 → Prompt Template → Structured Output`

이번 단계에서는 Tool Calling, LangGraph, LlamaIndex, Agent를 사용하지 않습니다.

## 0. 실행 전 확인

- Jupyter kernel: `skn25`
- Ollama 서버: `http://localhost:11434`
- 모델: `qwen3.6:27b`
- 필요한 패키지: `langchain`, `langchain-ollama`, `pydantic`

이 노트북은 API 키를 사용하지 않습니다. 모델은 내 컴퓨터의 Ollama 컨테이너에서 실행됩니다.

## 1. 필요한 클래스 불러오기

- `ChatOllama`: Ollama의 대화형 모델을 LangChain Chat Model 형태로 연결합니다.
- `ChatPromptTemplate`: 역할과 변수로 구성된 메시지 템플릿을 만듭니다.
- `BaseModel`, `Field`: 모델이 반환할 구조화된 데이터의 형태를 정의합니다.

In [ ]:
import json
from urllib.request import urlopen

from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

## 2. 연결할 모델 정보 정하기

나중에 다른 로컬 모델을 비교할 때는 아래 두 값만 바꾸면 됩니다. `temperature=0`은 같은 질문에 대한 응답의 변동을 줄이기 위한 설정입니다.

In [ ]:
OLLAMA_BASE_URL = "http://localhost:11434"
MODEL_NAME = "qwen3.6:27b"
TEMPERATURE = 0

## 3. Ollama 연결과 모델 존재 여부 확인

`localhost`는 Jupyter kernel이 실행되는 컴퓨터를 뜻합니다. 이 노트북은 호스트의 `skn25` 환경에서 실행하므로 Docker가 포트 `11434`를 호스트에 공개했다면 이 주소로 연결됩니다. 모델을 호출하기 전에 Ollama 서버와 모델 이름을 먼저 확인합니다.

In [ ]:
with urlopen(f"{OLLAMA_BASE_URL}/api/tags", timeout=5) as response:
    available_models = [model["name"] for model in json.load(response)["models"]]

if MODEL_NAME not in available_models:
    raise ValueError(f"{MODEL_NAME} 모델을 Ollama에서 찾지 못했습니다: {available_models}")

print(f"Ollama 연결 성공: {MODEL_NAME}")

## 4. Chat Model 객체 만들기

아직 모델을 실행하지는 않습니다. `llm`은 어디에 있는 어떤 모델을 어떤 설정으로 호출할지 담고 있는 객체입니다.

In [ ]:
llm = ChatOllama(
    model=MODEL_NAME,
    base_url=OLLAMA_BASE_URL,
    temperature=TEMPERATURE,
)

llm

## 5. 가장 작은 Chat Model 호출

`invoke()`는 입력 하나를 모델에 전달하고 응답 하나를 받는 가장 기본적인 LangChain 실행 방식입니다. 반환값은 단순 문자열이 아니라 메시지 객체입니다.

In [ ]:
response = llm.invoke("삼성전자가 어떤 회사인지 한 문장으로 설명해줘.")
print(response.content)

## 6. 응답 객체 확인

`content`에는 모델의 텍스트 답변이 있고, `response_metadata`에는 모델명·처리 시간처럼 실행을 이해하는 데 도움이 되는 정보가 들어갑니다. 실제 필드는 모델과 Ollama 버전에 따라 달라질 수 있습니다.

In [ ]:
print("응답 객체 타입:", type(response))
print("메타데이터:", response.response_metadata)

## 7. Prompt Template 만들기

앞의 문자열 질문은 간단하지만, 기업명·연도처럼 바뀌는 값을 매번 문자열로 조립하면 관리하기 어렵습니다. `ChatPromptTemplate`은 system 메시지와 사용자 메시지, 변수 자리를 분리합니다.

In [ ]:
financial_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 기업 분석을 돕는 어시스턴트다. 모르면 모른다고 답한다."),
        ("human", "기업명: {company}\n질문: {question}"),
    ]
)

## 8. 템플릿에 실제 값 넣어 보기

먼저 모델을 호출하지 않고, 템플릿이 어떤 메시지 묶음으로 바뀌는지 확인합니다. 이것이 Prompt를 분리해서 확인하는 이유입니다.

In [ ]:
prompt_value = financial_prompt.invoke(
    {
        "company": "삼성전자",
        "question": "2025년 매출과 영업이익을 확인할 때 무엇을 비교해야 해?",
    }
)

for message in prompt_value.messages:
    print(f"[{message.type}] {message.content}\n")

## 9. Prompt와 Model 연결하기

`|`는 앞 단계의 결과를 다음 단계의 입력으로 넘기는 LangChain의 연결 문법입니다. 지금은 `Prompt → Model`이라는 가장 작은 chain입니다.

In [ ]:
financial_chain = financial_prompt | llm

In [ ]:
chain_response = financial_chain.invoke(
    {
        "company": "삼성전자",
        "question": "2025년 매출과 영업이익을 확인할 때 무엇을 비교해야 해?",
    }
)
print(chain_response.content)

## 10. Structured Output용 데이터 형태 정의

LLM의 자유 텍스트를 다음 단계 코드가 바로 쓰기는 어렵습니다. Pydantic 모델로 기대하는 필드를 먼저 선언하면, 모델에게 그 형태의 결과를 요청할 수 있습니다.

여기서는 실제 OpenDART를 호출하지 않습니다. 사용자의 문장에서 조회 조건만 추출하는 예제입니다.

In [ ]:
class FinancialQuery(BaseModel):
    company: str = Field(description="조회할 기업명")
    business_year: int = Field(description="조회할 사업연도")
    metrics: list[str] = Field(description="조회하려는 재무 항목 목록")

## 11. Structured Output으로 호출하기

`with_structured_output()`은 같은 모델에 `FinancialQuery` 형태의 결과를 요청하도록 만든 새 Runnable입니다. `include_raw=True`로 설정하면 파싱 성공 여부와 원본 모델 응답도 함께 확인할 수 있습니다.

In [ ]:
structured_llm = llm.with_structured_output(
    FinancialQuery,
    include_raw=True,
)

In [ ]:
structured_response = structured_llm.invoke(
    "삼성전자의 2025년 매출액, 영업이익, 당기순이익을 조회해줘."
)

print("파싱 오류:", structured_response["parsing_error"])
print("원본 응답:", structured_response["raw"].content)

## 12. Python 딕셔너리로 변환하기

파싱 오류가 없을 때만 Pydantic 객체를 꺼냅니다. 그 뒤 다음 단계의 일반 Python 함수나 OpenDART 조회 함수에 넘기려면 `model_dump()`로 딕셔너리로 바꿀 수 있습니다.

In [ ]:
if structured_response["parsing_error"] is not None:
    raise RuntimeError("Structured Output 파싱에 실패했습니다. 원본 응답을 확인해 주세요.")

financial_query = structured_response["parsed"]
financial_query.model_dump()

## 선택: OpenAI API를 한 번 호출해 보기

로컬 Qwen과 코드가 얼마나 다른지 보기 위한 비교 예제입니다. `.env`의 `OPENAI_API_KEY`, `OPENAI_MODEL`을 실제 값으로 바꾼 뒤에만 실행하세요. 아래 두 코드 셀은 각각 API 요청 한 번을 발생시키며, 선택한 OpenAI 모델의 API 사용료가 발생합니다. 입력 문장은 OpenAI 외부 서비스로 전송됩니다. 비교 목적이라면 **둘 중 하나만** 실행해도 됩니다.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir
load_dotenv(project_root / ".env")

openai_api_key = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL")
if not openai_api_key or openai_api_key == "your_openai_api_key":
    raise ValueError(".env의 OPENAI_API_KEY를 실제 API 키로 바꿔 주세요.")
if not OPENAI_MODEL or OPENAI_MODEL == "your_openai_model_id":
    raise ValueError(".env의 OPENAI_MODEL을 사용할 수 있는 모델 ID로 바꿔 주세요.")

print(f"OpenAI 모델: {OPENAI_MODEL}")

### A. OpenAI 공식 SDK로 직접 한 번 호출

이 코드는 LangChain을 거치지 않는 가장 작은 OpenAI API 호출입니다. `OpenAI()`는 환경 변수의 `OPENAI_API_KEY`를 자동으로 읽습니다.

In [ ]:
from openai import OpenAI

client = OpenAI()
api_response = client.responses.create(
    model=OPENAI_MODEL,
    input="삼성전자가 어떤 회사인지 한 문장으로 설명해줘.",
)

print(api_response.output_text)

### B. 같은 요청을 LangChain 인터페이스로 호출

로컬 Qwen의 `ChatOllama`와 비교할 셀입니다. `ChatOpenAI`로 바뀌었을 뿐, `invoke()`와 응답의 `content`를 쓰는 흐름은 같습니다.

In [ ]:
from langchain_openai import ChatOpenAI

openai_llm = ChatOpenAI(model=OPENAI_MODEL)
openai_response = openai_llm.invoke("삼성전자가 어떤 회사인지 한 문장으로 설명해줘.")

print(openai_response.content)

## 이번 단계에서 확인할 개념

1. LangChain은 모델을 대신하는 것이 아니라, 서로 다른 모델을 비슷한 방식으로 호출하고 Prompt·출력 형식을 연결하게 돕는 도구입니다.
2. `invoke()`의 결과는 메시지 객체이며, 텍스트는 `content`에 있습니다.
3. Prompt Template은 변하는 입력값과 고정 지시문을 분리합니다.
4. Structured Output은 자연어 요청을 코드가 사용할 데이터 형태로 바꾸는 출발점입니다.
5. 로컬 모델과 OpenAI API는 연결 설정은 다르지만, LangChain을 쓰면 Prompt와 `invoke()` 흐름은 거의 같습니다.

다음 단계는 아직 진행하지 않습니다. Tool Calling을 시작할 때 OpenDART 함수를 LangChain Tool로 바꾸게 됩니다.